<center>
<img src="https://www.infnet.edu.br/infnet/wp-content/uploads/sites/18/2021/10/infnet-30-horizontal-padrao@300x-8-1024x265.png" width="60%"/>
</center>

# MBA em Engenharia de Dados: Big Data e IA
## Processamento de Big Data com Apache Spark e Spark SQL [26E3_2]
### Projeto da disciplina

### Variáveis dos volumes paths

In [0]:
CATALOG                     = 'instacart'

BRONZE_AISLES               = f'{CATALOG}.bronze.aisle'
BRONZE_DEPARTMENTS          = f'{CATALOG}.bronze.department'
BRONZE_ORDERS               = f'{CATALOG}.bronze.order'
BRONZE_PRODUCTS             = f'{CATALOG}.bronze.product'
BRONZE_ORDER_PRODUCTS_PRIOR = f'{CATALOG}.bronze.order_product_prior'
BRONZE_ORDER_PRODUCTS_TRAIN = f'{CATALOG}.bronze.order_product_train'

SILVER_AISLES               = f'{CATALOG}.silver.aisle'
SILVER_DEPARTMENTS          = f'{CATALOG}.silver.department'
SILVER_ORDERS               = f'{CATALOG}.silver.order'
SILVER_PRODUCTS             = f'{CATALOG}.silver.product'
SILVER_ORDER_PRODUCTS       = f'{CATALOG}.silver.order_product'

## CAMADA SILVER

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS instacart.silver;

CREATE TABLE IF NOT EXISTS instacart.silver.aisle (  
    aisle_id INTEGER,
    description STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.silver.department (  
    department_id INTEGER,
    description STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.silver.order (  
    order_id INTEGER,
    user_id INTEGER,
    eval_set STRING,
    order_number INTEGER,
    order_day_of_week INTEGER,
    order_hour_of_day  INTEGER,
    days_since_prior_order INTEGER
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.silver.product (  
    product_id INTEGER,
    description STRING,
    aisle_id INTEGER,
    department_id INTEGER
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS instacart.silver.order_product (  
    order_id INTEGER,
    product_id INTEGER,
    add_to_cart_order INTEGER,
    reordered INTEGER,
    eval_set STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
from pyspark.sql.functions import col, upper, lit

### Tabela: aisles
- Transformações:
  - Conversão do tipo de `aisle_id` para inteiro (`int`).
  - Remoção de registros duplicados com base na coluna `aisle`.
  - Padronização do texto de `aisle` para letras maiúsculas.
  - Renomeação da coluna `aisle` para `description`.
  - Preenchimento de valores nulos em `description` com o valor `"DESCONHECIDO"`.


In [0]:
aisles_df = spark.table(BRONZE_AISLES)
aisles_df.display()

In [0]:
aisles_df = (aisles_df
            .withColumn("aisle_id", col("aisle_id").cast("int"))
            .dropDuplicates(["aisle"])
            .withColumn("aisle", upper(col("aisle")))
            .withColumnRenamed("aisle", "description")
            .fillna({"description": "DESCONHECIDO"})
            )
aisles_df.display()

In [0]:
# Contagem de nulos
from pyspark.sql.functions import count, when
aisles_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in aisles_df.columns
]).show()

### Tabela: department
- Transformações:
  - Remoção de registros duplicados com base na coluna `department`.
  - Conversão do tipo de `department_id` para inteiro (`int`).
  - Padronização do texto de `department` para letras maiúsculas.
  - Renomeação da coluna `department` para `description`.
  - Preenchimento de valores nulos em `description` com o valor `"DESCONHECIDO"`.


In [0]:
departments_df = spark.table(BRONZE_DEPARTMENTS)
departments_df.display()

In [0]:
departments_df = (departments_df
            .dropDuplicates(["department"])
            .withColumn("department_id", col("department_id").cast("int"))
            .withColumn("department", upper(col("department")))
            .withColumnRenamed("department", "description")
            .fillna({"description": "DESCONHECIDO"})
            )
departments_df.display()

In [0]:
# Contagem de nulos
from pyspark.sql.functions import count, when
departments_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in departments_df.columns
]).show()

### Tabela: order
- Transformações:
  - Conversão dos tipos de `order_id`, `user_id`, `order_number`, `order_dow` e `order_hour_of_day` para inteiro (`int`).
  - Conversão do tipo de `days_since_prior_order` para ponto flutuante (`float`).
  - Padronização do texto de `eval_set` para letras maiúsculas.
  - Renomeação da coluna `order_dow` para `order_day_of_week`.
  - Preenchimento de valores nulos: `eval_set` com `"DESCONHECIDO"` e `days_since_prior_order` com `-1.0`.


In [0]:
orders_df = spark.table(BRONZE_ORDERS)
orders_df.display()

In [0]:
orders_df = (orders_df
            .withColumn("order_id", col("order_id").cast("int"))
            .withColumn("user_id", col("user_id").cast("int"))
            .withColumn("order_number", col("order_number").cast("int"))
            .withColumn("order_dow", col("order_dow").cast("int"))
            .withColumn("order_hour_of_day", col("order_hour_of_day").cast("int"))
            .withColumn("days_since_prior_order", col("days_since_prior_order").cast("float"))
            .withColumn("eval_set", upper(col("eval_set")))
            .withColumnRenamed("order_dow", "order_day_of_week")
            .fillna({"eval_set": "DESCONHECIDO", "days_since_prior_order": -1.0})
            )
orders_df.display()

In [0]:
# Contagem de nulos
from pyspark.sql.functions import count, when
orders_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in orders_df.columns
]).show()

### Tabela: product
- Transformações:
  - Conversão dos tipos de `product_id`, `aisle_id` e `department_id` para inteiro (`int`).
  - Renomeação da coluna `product_name` para `description`.
  - Padronização do texto de `description` para letras maiúsculas.
  - Preenchimento de valores nulos em `description` com o valor `"DESCONHECIDO"`.


In [0]:
products_df = spark.table(BRONZE_PRODUCTS)
products_df.display()

In [0]:
products_df = (products_df
            .withColumn("product_id", col("product_id").cast("int"))
            .withColumn("aisle_id", col("aisle_id").cast("int"))
            .withColumn("department_id", col("department_id").cast("int"))
            .withColumnRenamed("product_name", "description")
            .withColumn("description", upper(col("description")))
            .fillna({"description": "DESCONHECIDO"})
            )
products_df.display()

In [0]:
# Contagem de nulos
from pyspark.sql.functions import count, when
products_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in products_df.columns
]).show()

### Tabela: order_product
- Transformações:
  - Conversão dos tipos de `order_id`, `product_id` e `add_to_cart_order` para inteiro (`int`), aplicada separadamente aos conjuntos `order_product_prior` e `order_product_train`.
  - Criação da coluna `eval_set` com valor fixo `"PRIOR"` para os registros de `order_product_prior` e `"TRAIN"` para os registros de `order_product_train`.
  - União (`union`) dos dois conjuntos transformados em um único DataFrame `order_products_df`.


In [0]:
order_products_prior_df = spark.table(BRONZE_ORDER_PRODUCTS_PRIOR)
order_products_prior_df.display()

In [0]:
order_products_prior_df = (order_products_prior_df
            .withColumn("order_id", col("order_id").cast("int"))
            .withColumn("product_id", col("product_id").cast("int"))
            .withColumn("add_to_cart_order", col("add_to_cart_order").cast("int"))
            .withColumn("eval_set", lit("PRIOR"))
            )
order_products_prior_df.display()

In [0]:
# Contagem de nulos
from pyspark.sql.functions import count, when
order_products_prior_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in order_products_prior_df.columns
]).show()

In [0]:
order_products_train_df = spark.table(BRONZE_ORDER_PRODUCTS_TRAIN)
order_products_train_df.display()

In [0]:
order_products_train_df = (order_products_train_df
            .withColumn("order_id", col("order_id").cast("int"))
            .withColumn("product_id", col("product_id").cast("int"))
            .withColumn("add_to_cart_order", col("add_to_cart_order").cast("int"))
            .withColumn("eval_set", lit("TRAIN"))
            )
order_products_train_df.display()

In [0]:
order_products_df = order_products_train_df.union(order_products_prior_df)
order_products_df.display()

In [0]:
merge_data(df=aisles_df, layer_path=SILVER_AISLES, merge_keys=['aisle_id'])
merge_data(df=departments_df, layer_path=SILVER_DEPARTMENTS, merge_keys=['department_id'])
merge_data(df=orders_df, layer_path=SILVER_ORDERS, merge_keys=['order_id'])
merge_data(df=products_df, layer_path=SILVER_PRODUCTS, merge_keys=['product_id'])
merge_data(df=order_products_df, layer_path=SILVER_ORDER_PRODUCTS, merge_keys=order_products_df.columns)